# Relance des runs faits avec l'ancien `train.py` (Colab, GPU)

Les tout premiers runs MLP ont été faits avec une ancienne version de `train.py`, qui mesurait la loss d'entraînement sur tout le train (au lieu de 10 000 images fixes) et n'enregistrait pas `train_acc`. Pour que **tous** les runs de `fashion_mnist.ipynb` viennent de la même version du code, on les refait.

1. On range dans `resultats/ancien/` tous les CSV qui n'ont pas de colonne `train_acc` (= ancienne version).
2. On relance exactement les runs dont `fashion_mnist.ipynb` a besoin : calibration du MLP, grille MLP à 20 epochs, contrôle MLP largeur 64. Les runs déjà à jour sont sautés.
3. On ajoute la durée 0.05 au CNN largeur 8 à 20 epochs, pour qu'il ait la même grille que les autres.

Environ 65 runs, presque tous des petits MLP. Si Colab coupe, on relance tout : ça reprend où ça s'était arrêté.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout
# si le pull refuse à cause de modifs locales : !git stash  puis relancer la cellule
!git pull

In [ ]:
import os, glob, importlib
import pandas as pd
import torch
from torchvision import datasets

import config, models, train
importlib.reload(config); importlib.reload(models); importlib.reload(train)
from config import BATCH, P, EARLY_GRID
from models import mlp, cnn
from train import entrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, "| grille :", EARLY_GRID)

## 1. On range les CSV de l'ancienne version

In [ ]:
os.makedirs("resultats/ancien", exist_ok=True)
anciens = [f for f in glob.glob("resultats/*.csv") if "train_acc" not in pd.read_csv(f, nrows=1).columns]
for f in anciens:
    os.replace(f, f"resultats/ancien/{os.path.basename(f)}")
print(len(anciens), "fichiers rangés dans resultats/ancien/")
print(sorted({os.path.basename(f).split("_frac")[0] for f in anciens}))

In [ ]:
tr = datasets.FashionMNIST("./data", train=True, download=True)
te = datasets.FashionMNIST("./data", train=False, download=True)
Xtr, Xte = tr.data.float().unsqueeze(1) / 255, te.data.float().unsqueeze(1) / 255
ytr, yte = tr.targets, te.targets

# même split et même normalisation que dans fashion_mnist.ipynb
perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(0))
Xva, yva = Xtr[perm[:10000]], ytr[perm[:10000]]
Xtr, ytr = Xtr[perm[10000:]], ytr[perm[10000:]]
mu, sd = Xtr.mean(), Xtr.std()
Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

images = {k: v.to(device) for k, v in dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte).items()}   # CNN
vecteurs = {k: (v.flatten(1) if k.startswith("X") else v) for k, v in images.items()}                        # MLP

In [ ]:
# même fonction que dans fashion_mnist.ipynb : un run = un CSV, sauté s'il existe déjà
CADRES = {
    "mlp_fmnist": (lambda L: mlp(784, 10, couche_cache=L), vecteurs),
    "cnn_fmnist": (lambda L: cnn(1, 10, largeur=L), images),
}

def fichier(cadre, L, E, frac, seed, dossier="resultats"):
    return f"{dossier}/{cadre}_w{L}_e{E}_frac{frac}_s{seed}.csv"

def lancer(cadre, L, E, frac, seed, p=P, dossier="resultats"):
    f = fichier(cadre, L, E, frac, seed, dossier)
    if os.path.exists(f):
        return
    creer, data = CADRES[cadre]
    torch.manual_seed(seed)
    model, opt = creer(L)
    model.to(device)
    hist = entrainer(model, opt, data, E, t_early=round(frac * E), p=p, batch=BATCH)
    hist.assign(cadre=cadre, width=L, epochs=E, frac=frac, seed=seed).to_csv(f, index=False)
    print(f, " val acc", round(hist.val_acc.tail(5).mean(), 4))

## 2. Calibration du MLP (largeurs 16 / 32 / 64, budgets 10 / 20 / 50)

In [ ]:
for L in [16, 32, 64]:
    for E in [10, 20, 50]:
        for frac in [0, 1]:
            for seed in [0, 1]:
                lancer("mlp_fmnist", L, E, frac, seed)

## 3. Grille MLP largeur 16 à 20 epochs (toutes les durées, 5 seeds)

In [ ]:
for frac in EARLY_GRID:
    for seed in [0, 1, 2, 3, 4]:
        lancer("mlp_fmnist", 16, 20, frac, seed)

## 4. Contrôle MLP largeur 64 à 50 epochs (sans dropout, early 10 %, dropout classique, 3 seeds)

In [ ]:
for frac in [0, 0.1, 1]:
    for seed in [0, 1, 2]:
        lancer("mlp_fmnist", 64, 50, frac, seed)

## 5. CNN largeur 8 à 20 epochs : la durée 0.05 qui manquait

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    lancer("cnn_fmnist", 8, 20, 0.05, seed)

## Vérification : plus aucun run de l'ancienne version dans resultats/

In [ ]:
restants = [f for f in glob.glob("resultats/*.csv") if "train_acc" not in pd.read_csv(f, nrows=1).columns]
print("CSV sans train_acc encore dans resultats/ :", len(restants))

attendus = ([fichier("mlp_fmnist", L, E, f, s) for L in [16, 32, 64] for E in [10, 20, 50] for f in [0, 1] for s in [0, 1]]
            + [fichier("mlp_fmnist", 16, 20, f, s) for f in EARLY_GRID for s in range(5)]
            + [fichier("mlp_fmnist", 64, 50, f, s) for f in [0, 0.1, 1] for s in range(3)]
            + [fichier("cnn_fmnist", 8, 20, 0.05, s) for s in range(5)])
attendus = sorted(set(attendus))
print("runs attendus présents :", sum(os.path.exists(f) for f in attendus), "/", len(attendus))

## Sauvegarde sur GitHub (token dans les secrets Colab, jamais dans le code)

In [ ]:
!git config user.name "Felix-prjcts"
!git add -A resultats/
!git commit -m "relance des runs MLP de l'ancienne version de train.py + CNN w8 e20 frac 0.05"

token = userdata.get("GITHUB_TOKEN")
!git push -q https://{token}@github.com/Felix-prjcts/early-dropout-study.git HEAD 2>&1 | sed "s/{token}/***/g"